In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, group
from pyspark.sql import functions as F

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
cat, src = cfg["catalog"], group.source(cfg, env)
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print(("PASS " if ok else "FAIL ") + name + (f" -> {detail}" if detail and not ok else ""))

g = lambda t: spark.table(f"{cat}.gold.{t}")
f = lambda t: spark.table(f"{src}.gold.{t}")
print(cat, "merge control, FQF from", src)

In [0]:
from datetime import date
log, backlog = spark.table(f"{cat}.ops.merge_log"), spark.table(f"{cat}.ops.merge_backlog")
last = log.filter(F.col("env") == env).orderBy(F.col("run_ts").desc()).first()
check("the group run wrote a merge_log row", last is not None)
print("INFO last run:", {k: str(last[k]) for k in ("run_ts", "bhg_data_to", "fqf_data_to", "new_fqf_days", "backlog_days", "status")})

check("data dates are not in the future", last["bhg_data_to"] <= date.today() and last["fqf_data_to"] <= date.today())
check("status matches the backlog", (last["status"] == "complete") == (last["backlog_days"] == 0), last["status"])
waiting = backlog.filter("status = 'waiting'")
check("backlog waiting count = last log row", waiting.count() == last["backlog_days"], waiting.count())
check("nothing waits that FQF has already delivered",
      waiting.filter(F.col("business_date") <= F.lit(last["fqf_data_to"])).count() == 0)
check("one backlog row per day", backlog.select("business_date").distinct().count() == backlog.count())

# probe: a day FQF has already delivered is marked merged on the next run (then the probe is removed)
spark.sql(f"INSERT INTO {cat}.ops.merge_backlog VALUES (DATE'2000-01-01', current_timestamp(), 'waiting', NULL)")
row = group.merge_control(spark, cfg, env, {"probe": 0})
probe = backlog.filter("business_date = DATE'2000-01-01'").select("status").first()[0]
check("probe: a delivered day is marked merged", probe == "merged", probe)
check("second run in a row merges no new FQF days", row["new_fqf_days"] == 0, row["new_fqf_days"])
spark.sql(f"DELETE FROM {cat}.ops.merge_backlog WHERE business_date = DATE'2000-01-01'")
spark.sql(f"DELETE FROM {cat}.ops.merge_log WHERE tables = \"{{'probe': 0}}\"")

today = int(date.today().strftime("%Y%m%d"))
fb = g("fact_bhg_event").filter(F.col("date_key") > today).count()
ff = f("fact_fqf_event").filter((F.col("date_key") > today) & (F.col("event_type") != "emi_due")).count()
print(f"INFO events dated in the future (bad source dates, ignored for merge dates): BHG {fb}, FQF {ff}")

print(f"\nt25_merge_control: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t25_merge_control failed")